<div style="background:#123B63;color:white;padding:14px 18px;border-radius:6px">
<b>CSE 816 &mdash; Machine Learning Lab</b> &nbsp;&middot;&nbsp; Department of CSE, University of Chittagong<br>
<span style="font-size:90%">Module 2 &middot; Week 5 &middot; Part 1 of 4 &nbsp;&middot;&nbsp; 60 minutes</span>
</div>

# The ReLU Activation

Every unit in Week 4 used the sigmoid. This part replaces it, in the hidden layers, with the
**rectified linear unit** $g(z) = \max(0, z)$, and asks two questions with numbers rather than
words: what changes when you swap the activation, and what goes wrong if you leave the
activation out altogether?

On the way, you will look inside `compile`: the loss you name there is a formula you can
compute yourself, and you will check that TensorFlow computes the same number.

Part 2 adds softmax for many classes; Part 3 opens up back-propagation and Adam; Part 4 is the
practice lab.

**Companion theory lecture:** CSE 815, Week 5, Part 1 (Training Neural Networks).

## What you will be able to do

1. Implement sigmoid, ReLU and linear activations in NumPy and match them to Keras.
2. Reproduce the lecture's worked example with a ReLU hidden layer, in NumPy and TensorFlow.
3. Compute binary cross-entropy and mean squared error by hand and confirm that
   `model.evaluate` returns the same cost.
4. Show that a stack of linear layers collapses to one layer, and compute that layer.
5. Build a piecewise-linear function from ReLU units by hand, then let a network learn it.

---

## 1. Setup

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")   # hide TensorFlow's start-up chatter

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Dense

%matplotlib inline
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

np.set_printoptions(precision=4, suppress=True)
keras.utils.set_random_seed(816)      # seeds Python, NumPy and TensorFlow together

BLUE, TEAL, AMBER, ROSE, GRAY = "#123B63", "#1B7F79", "#C97B17", "#9B2C4B", "#4A5560"

print("NumPy", np.__version__, "  TensorFlow", tf.__version__)

---

## 2. Three activations

| | $g(z)$ | range | typical use |
|---|---|---|---|
| sigmoid | $1/(1+e^{-z})$ | $(0, 1)$ | output of a binary classifier |
| ReLU | $\max(0, z)$ | $[0, \infty)$ | hidden layers; non-negative outputs |
| linear | $z$ | $(-\infty, \infty)$ | output of a regression |

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def relu(z):
    return np.maximum(0.0, z)


def linear(z):
    return z


z = np.array([-3.0, -0.5, 0.0, 0.5, 3.0])
print("z       :", z)
print("sigmoid :", sigmoid(z))
print("relu    :", relu(z))
print("linear  :", linear(z))

assert np.array_equal(relu(z), [0, 0, 0, 0.5, 3])
for mine, name in [(sigmoid, "sigmoid"), (relu, "relu"), (linear, "linear")]:
    theirs = keras.activations.get(name)(z.astype("float32"))
    assert np.allclose(mine(z), np.asarray(theirs), atol=1e-6), name
print("\nAll three agree with keras.activations.")

Look at the activations **and their slopes**. The slope is what gradient descent follows, so a
flat activation means slow learning.

In [ ]:
zz = np.linspace(-6, 6, 601)
eps = 1e-4
slope = lambda g: (g(zz + eps) - g(zz - eps)) / (2 * eps)   # central difference

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
for g, c, name in [(sigmoid, BLUE, "sigmoid"), (relu, TEAL, "ReLU"), (linear, AMBER, "linear")]:
    axes[0].plot(zz, g(zz), c=c, lw=2, label=name)
    axes[1].plot(zz, slope(g), c=c, lw=2, label=name)
axes[0].set_ylim(-1.5, 4); axes[0].set_title("g(z)", fontsize=10)
axes[1].set_ylim(-0.1, 1.15); axes[1].set_title("slope g'(z)", fontsize=10)
for ax in axes:
    ax.set_xlabel("z"); ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

s = slope(sigmoid)
print(f"largest sigmoid slope: {s.max():.4f}  (at z = 0)")
print(f"sigmoid slope at z = 5: {s[zz == 5][0]:.4f}")
assert np.isclose(s.max(), 0.25, atol=1e-6)

The sigmoid's slope is never more than 0.25 and is almost zero once $|z| > 5$: a sigmoid unit
that is confidently on or off barely moves under gradient descent. ReLU's slope is exactly 1
for every positive $z$. That is the main reason it is the default in hidden layers.

---

## 3. The lecture's network, with a ReLU hidden layer

The $2 \to 3 \to 1$ network from Week 4 keeps its weights; only the hidden activation changes.
The output stays sigmoid, because the target is still 0 or 1.

$$\mathbf{W}^{[1]} = \begin{bmatrix}1&-3&5\\2&4&-6\end{bmatrix},\quad
  \mathbf{b}^{[1]} = [-1,\ 1,\ 2],\qquad
  \mathbf{w}^{[2]} = [2,\ -1,\ -3],\quad b^{[2]} = 0.5$$

In [ ]:
W1 = np.array([[1, -3,  5],
               [2,  4, -6]], dtype=float)
b1 = np.array([-1, 1, 2], dtype=float)
W2 = np.array([[2], [-1], [-3]], dtype=float)
b2 = np.array([0.5])
X_demo = np.array([[1.0, 2.0],
                   [-1.0, -1.0]])


def forward(X, hidden):
    """Two-layer forward propagation with the given hidden activation; sigmoid output."""
    Z1 = X @ W1 + b1
    A1 = hidden(Z1)
    return Z1, A1, sigmoid(A1 @ W2 + b2)


for hidden in (sigmoid, relu):
    Z1, A1, A2 = forward(X_demo, hidden)
    print(f"--- hidden activation: {hidden.__name__}")
    print("Z1 =\n", Z1, "\nA1 =\n", A1, "\nA2 =", A2.ravel(), "\n")

_, A1_relu, A2_relu = forward(X_demo, relu)
assert np.array_equal(A1_relu, [[4, 6, 0], [0, 0, 3]])
assert np.allclose(A2_relu.ravel(), [0.9241, 0.0002], atol=5e-5)
print("Matches the lecture: 0.9241 and 0.0002 (Week 4's sigmoid version: 0.8094, 0.0562).")

The same $\mathbf{Z}^{[1]}$, different $\mathbf{A}^{[1]}$. ReLU zeros the negative entries and
passes the positive ones through unsquashed, so the output is more confident in both
directions.

Now the same network in TensorFlow: one word changes, `"sigmoid"` becomes `"relu"`.

In [ ]:
net_relu = Sequential([
    Input(shape=(2,)),
    Dense(units=3, activation="relu", name="L1"),
    Dense(units=1, activation="sigmoid", name="L2"),
])
net_relu.get_layer("L1").set_weights([W1, b1])
net_relu.get_layer("L2").set_weights([W2, b2])

p_tf = net_relu.predict(X_demo, verbose=0)
print("TensorFlow:", p_tf.ravel())
assert np.allclose(p_tf, A2_relu, atol=1e-6)
print("NumPy and TensorFlow agree.")

---

## 4. Inside `compile`: the loss is a formula

`model.compile(loss=...)` names the function that `fit` minimises. For a 0/1 target it is
**binary cross-entropy** &mdash; the logistic loss of Week 3:

$$L\bigl(f(\mathbf{x}), y\bigr) = -y\log f(\mathbf{x}) - (1-y)\log\bigl(1 - f(\mathbf{x})\bigr),
  \qquad J = \frac{1}{m}\sum_{i=1}^{m} L\bigl(f(\mathbf{x}^{(i)}), y^{(i)}\bigr).$$

`model.evaluate` computes $J$ on whatever data you give it. Compute it yourself and compare.

In [ ]:
def bce(f, y):
    """Mean binary cross-entropy of predictions f against labels y."""
    return float(np.mean(-y * np.log(f) - (1 - y) * np.log(1 - f)))


y_demo = np.array([1.0, 0.0])
f = A2_relu.ravel()
per_example = -y_demo * np.log(f) - (1 - y_demo) * np.log(1 - f)
print("loss per example:", per_example)
print(f"cost J (mine)   : {bce(f, y_demo):.6f}")

net_relu.compile(loss=keras.losses.BinaryCrossentropy())
J_tf = net_relu.evaluate(X_demo, y_demo, verbose=0)
print(f"cost J (Keras)  : {J_tf:.6f}")
assert np.isclose(bce(f, y_demo), J_tf, atol=1e-5)

# The lecture's table, for Week 4's sigmoid network: f = 0.8094 and f = 0.0562.
f4 = np.array([0.8094, 0.0562])
assert np.allclose(-np.log(f4), [0.2115, 2.8788], atol=5e-5)        # if y = 1
assert np.allclose(-np.log(1 - f4), [1.6576, 0.0578], atol=5e-5)    # if y = 0
print("\nThe lecture's loss table checks out: 0.2115, 2.8788 (y = 1); 1.6576, 0.0578 (y = 0).")

For a numeric target the loss is the squared error, `keras.losses.MeanSquaredError()`. Note one
convention difference: Keras's MSE is $\frac{1}{m}\sum (f - y)^2$, *without* the
$\frac{1}{2}$ we used in Weeks 1&ndash;3. The factor halves the gradient, which only rescales the
learning rate; it does not move the minimum.

In [ ]:
f_reg = np.array([2.5, 0.0, 4.0])
y_reg = np.array([3.0, -1.0, 4.0])
mse_keras = float(keras.losses.MeanSquaredError()(y_reg, f_reg))
print(f"Keras MSE           : {mse_keras:.4f}")
print(f"mean (f - y)^2      : {np.mean((f_reg - y_reg) ** 2):.4f}")
print(f"our Week 1 cost J   : {np.mean((f_reg - y_reg) ** 2) / 2:.4f}")
assert np.isclose(mse_keras, np.mean((f_reg - y_reg) ** 2))

---

## 5. Without a non-linear activation, depth does nothing

Give every layer the linear activation $g(z) = z$. Then

$$\mathbf{a}^{[2]} = (\mathbf{x}\mathbf{W}^{[1]} + \mathbf{b}^{[1]})\mathbf{W}^{[2]} + \mathbf{b}^{[2]}
  = \mathbf{x}\underbrace{\mathbf{W}^{[1]}\mathbf{W}^{[2]}}_{\mathbf{W}'}
  + \underbrace{\mathbf{b}^{[1]}\mathbf{W}^{[2]} + \mathbf{b}^{[2]}}_{\mathbf{b}'}$$

&mdash; one linear layer. The lecture computed $\mathbf{W}' = [-10,\ 18]^\top$ and
$b' = -8.5$ for our network.

In [ ]:
net_lin = Sequential([
    Input(shape=(2,)),
    Dense(units=3, activation="linear", name="L1"),
    Dense(units=1, activation="linear", name="L2"),
])
net_lin.get_layer("L1").set_weights([W1, b1])
net_lin.get_layer("L2").set_weights([W2, b2])

W_eq = W1 @ W2
b_eq = b1 @ W2 + b2
print("W' =", W_eq.ravel(), "  b' =", b_eq)
assert np.allclose(W_eq.ravel(), [-10, 18]) and np.allclose(b_eq, [-8.5])

two_layers = net_lin.predict(X_demo, verbose=0).ravel()
one_layer = (X_demo @ W_eq + b_eq).ravel()
print("two linear layers:", two_layers, "   one layer:", one_layer)
assert np.allclose(two_layers, [17.5, -16.5], atol=1e-4)

# Not a coincidence of two inputs: 1000 random points agree too.
X_rand = np.random.default_rng(0).normal(size=(1000, 2)) * 5
assert np.allclose(net_lin.predict(X_rand, verbose=0), X_rand @ W_eq + b_eq, atol=1e-3)
print("...and on 1000 random inputs. The 'network' is a single linear function.")

So a network with linear hidden layers and a linear output is linear regression; with a sigmoid
output it is logistic regression. The extra layers add parameters, but no ability.

---

## 6. What ReLU buys: functions with corners

Each ReLU unit contributes one **hinge** &mdash; a corner where its $z$ crosses zero. A linear
output unit adds hinges with weights. The lecture's example is

$$f(x) = r(x) - 2\,r(x - 1) + 2\,r(x - 2.5), \qquad r(z) = \max(0, z).$$

That is a $1 \to 3 \to 1$ network with ReLU hidden units and a linear output, whose weights you
can read off the formula: hidden unit $j$ computes $r(1 \cdot x + b_j)$ with
$b = [0, -1, -2.5]$, and the output weights are $[1, -2, 2]$.

In [ ]:
def target(x):
    return relu(x) - 2 * relu(x - 1) + 2 * relu(x - 2.5)


hinge = Sequential([
    Input(shape=(1,)),
    Dense(units=3, activation="relu", name="hidden"),
    Dense(units=1, activation="linear", name="out"),
])
hinge.get_layer("hidden").set_weights([np.array([[1.0, 1.0, 1.0]]), np.array([0.0, -1.0, -2.5])])
hinge.get_layer("out").set_weights([np.array([[1.0], [-2.0], [2.0]]), np.array([0.0])])

x_grid = np.linspace(-0.5, 4.0, 451).reshape(-1, 1)
f_hand = hinge.predict(x_grid, verbose=0)
assert np.allclose(f_hand.ravel(), target(x_grid.ravel()), atol=1e-5)

for x0 in [0.0, 1.0, 2.5, 4.0]:
    print(f"f({x0}) = {target(x0):.2f}")

fig, ax = plt.subplots(figsize=(6.0, 3.4))
for j, (w, b, c) in enumerate([(1, 0, BLUE), (-2, -1, TEAL), (2, -2.5, AMBER)]):
    ax.plot(x_grid, w * relu(x_grid + b), c=c, ls="--", lw=1, label=f"{w:+d} r(x{b:+g})")
ax.plot(x_grid, f_hand, c=ROSE, lw=2.5, label="sum = network output")
ax.set_ylim(-1.5, 2.5); ax.set_xlabel("x"); ax.legend(fontsize=8)
ax.set_title("Three ReLU units, added with weights", fontsize=10)
plt.show()

Slopes $1$, then $1 - 2 = -1$, then $-1 + 2 = 1$: every change of slope is a hidden unit. With
enough units, sums of hinges can follow any continuous curve on a bounded interval.

### Can a network *learn* it?

Now forget the hand-set weights. Give 200 samples of $f$ to two networks with 16 hidden units
each &mdash; one with ReLU, one with a linear hidden layer &mdash; and train both with the
squared-error loss.

In [ ]:
x_tr = np.linspace(-0.5, 4.0, 200).reshape(-1, 1)
y_tr = target(x_tr.ravel())


def fit_regressor(hidden_activation, seed=816, epochs=300):
    keras.utils.set_random_seed(seed)
    model = Sequential([
        Input(shape=(1,)),
        Dense(units=16, activation=hidden_activation),
        Dense(units=1, activation="linear"),
    ])
    model.compile(loss=keras.losses.MeanSquaredError(),
                  optimizer=keras.optimizers.Adam(learning_rate=0.01))
    history = model.fit(x_tr, y_tr, epochs=epochs, verbose=0)
    return model, history.history["loss"]


relu_net, relu_loss = fit_regressor("relu")
lin_net, lin_loss = fit_regressor("linear")

# The best any linear function can do: the least-squares line.
slope_ls, icpt_ls = np.polyfit(x_tr.ravel(), y_tr, 1)
mse_line = np.mean((slope_ls * x_tr.ravel() + icpt_ls - y_tr) ** 2)

print(f"final MSE, ReLU hidden layer   : {relu_loss[-1]:.5f}")
print(f"final MSE, linear hidden layer : {lin_loss[-1]:.5f}")
print(f"MSE of the least-squares line  : {mse_line:.5f}")
assert relu_loss[-1] < 0.01
assert lin_loss[-1] > 0.9 * mse_line     # cannot beat the best straight line

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].plot(x_tr, y_tr, c=GRAY, lw=4, alpha=0.35, label="target f(x)")
axes[0].plot(x_tr, relu_net.predict(x_tr, verbose=0), c=TEAL, lw=2, label="ReLU network")
axes[0].plot(x_tr, lin_net.predict(x_tr, verbose=0), c=AMBER, lw=2, label="linear network")
axes[0].set_xlabel("x"); axes[0].legend(fontsize=8)
axes[0].set_title("Fits after 300 epochs", fontsize=10)
axes[1].plot(relu_loss, c=TEAL, lw=2, label="ReLU")
axes[1].plot(lin_loss, c=AMBER, lw=2, label="linear")
axes[1].set_yscale("log"); axes[1].set_xlabel("epoch"); axes[1].set_ylabel("training MSE")
axes[1].legend(fontsize=8); axes[1].set_title("Learning curves", fontsize=10)
plt.tight_layout()
plt.show()

The linear network, with 16 hidden units and 49 parameters, ends up as a straight line: the
same line `np.polyfit` finds in one call. The ReLU network finds the corners.

> **Dead units.** A ReLU unit whose $z$ is negative for *every* training example outputs 0 and
> has slope 0, so gradient descent never changes it. Count how many of the 16 hidden units in
> `relu_net` are dead on the training inputs:

In [ ]:
W_h, b_h = relu_net.layers[0].get_weights()
A_hidden = relu(x_tr @ W_h + b_h)                # (200, 16)
alive = (A_hidden > 0).any(axis=0)
print(f"units active on at least one training input: {int(alive.sum())} of {alive.size}")
print("hinge positions x = -b/w of the live units:",
      np.sort((-b_h / W_h[0])[alive]).round(2))

In the reference run none of the 16 is dead, but a unit whose hinge lies outside $[-0.5, 4]$
contributes no corner: it is either always on (a straight segment) or would be always off. The
hinges that matter cluster near $x = 0$, $1$ and $2.5$ &mdash; exactly where the target has its
corners. Nobody told the network where they were.

### Checkpoint 1

Using `forward` from section 3 (ReLU hidden layer), compute $\mathbf{z}^{[1]}$,
$\mathbf{a}^{[1]}$, $z^{[2]}$ and $a^{[2]}$ for $\mathbf{x} = [0, 0]$ and for
$\mathbf{x} = [2, 0]$ (lecture, descriptive question 9). Confirm with `net_relu.predict`, and
state each prediction $\hat{y}$.

*Expected:* for $[0, 0]$, $\mathbf{a}^{[1]} = [0, 1, 2]$, $z^{[2]} = -6.5$,
$a^{[2]} \approx$ `0.0015`; for $[2, 0]$, $\mathbf{a}^{[1]} = [1, 0, 12]$, $z^{[2]} = -33.5$,
$a^{[2]} \approx 2.83 \times 10^{-15}$. Both predict $\hat{y} = 0$. The second output is
tiny but not zero: float32 can represent numbers down to about $10^{-38}$.

In [ ]:
# Your code here

### Checkpoint 2

Design by hand a $1 \to 2 \to 1$ network &mdash; ReLU hidden layer, linear output &mdash; that
outputs $0$ for $x < 0$, rises with slope 2 up to $x = 1$, and is flat at $2$ after that
(lecture, design question 4). Load your weights into a `Sequential` model and `assert` that it
matches $2\,r(x) - 2\,r(x-1)$ on `np.linspace(-2, 3, 101)`.

*Expected:* outputs `0`, `1`, `2`, `2` at $x = -1,\ 0.5,\ 1,\ 3$.

In [ ]:
# Your code here

---

## 7. Recap

- ReLU, $\max(0, z)$, is zero for negative inputs and the identity for positive ones. Its slope
  is 1 wherever it is active; the sigmoid's slope never exceeds 0.25.
- Changing the lecture network's hidden activation to ReLU keeps $\mathbf{Z}^{[1]}$ and
  changes the outputs to `0.9241` and `0.0002`.
- The loss in `compile` is a formula. `BinaryCrossentropy` is Week 3's logistic loss;
  `MeanSquaredError` is the squared error without the $\frac{1}{2}$. `model.evaluate` returns
  the same cost you compute by hand.
- Linear layers compose to one linear layer: $\mathbf{W}' = \mathbf{W}^{[1]}\mathbf{W}^{[2]}$,
  $\mathbf{b}' = \mathbf{b}^{[1]}\mathbf{W}^{[2]} + \mathbf{b}^{[2]}$. Trained on a bent target,
  a linear network can do no better than the least-squares line.
- Each ReLU unit adds one hinge. Hand-set or learned, sums of hinges fit functions with corners.

### Exercises to hand in

1. **Sigmoid versus ReLU, measured.** Train the hinge regressor of section 6 with
   `"sigmoid"` hidden units, same seed and epochs. Plot the three learning curves together and
   report the epoch at which each first drops below MSE `0.05`. Relate the result to the slope
   plot of section 2.
2. **How many units?** Repeat the ReLU fit with 2, 3, 4, 8 and 32 hidden units, three seeds
   each. Tabulate the final MSE. What is the smallest width that reliably fits the target, and
   why can it not be smaller than 3?
3. **Collapse, in general.** Write `collapse(model)` that takes any `Sequential` model whose
   `Dense` layers are all linear and returns the equivalent single $(\mathbf{W}', \mathbf{b}')$.
   Test it on a random $5 \to 7 \to 4 \to 2$ linear model.
4. **Leaky ReLU.** `keras.activations.leaky_relu` uses $g(z) = z$ for $z > 0$ and
   $0.2z$ otherwise. Can a leaky-ReLU unit be dead? Repeat the dead-unit count of section 6 with
   it, and explain the difference.

### Next

**Part 2 &mdash; Softmax and multiclass classification:** an output layer for $N$ classes, and
why TensorFlow wants logits rather than probabilities.